In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_3.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common data loading, windowing, training, visualisation, and experiment
# tracking utilities shared across all technique files in this exercise.
#
# This module is NOT meant to be run standalone — import it from the
# technique files (01_vanilla_rnn.py, 02_lstm.py, etc.).
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec

# ── Constants ───────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "stocks"
OUTPUT_DIR = REPO_ROOT / "outputs" / "mlfp05" / "ex_3"

TICKERS = {
    "^STI": "Straits Times Index",
    "DBS.SI": "DBS Group",
    "9988.HK": "Alibaba HK",
    "AAPL": "Apple",
    "005930.KS": "Samsung",
    "7203.T": "Toyota",
}

SEQ_LEN = 20  # 20-day lookback (4 trading weeks)
FORECAST_HORIZON = 5  # predict next 5 days
FEATURES = ["Close", "High", "Low", "Volume"]
HIDDEN_DIM = 64
EPOCHS = 15
LR = 1e-3
CLIP = 1.0
BATCH_SIZE = 64


def init_environment() -> torch.device:
    """Set up environment, seeds, device, and output directories."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ── Data Loading ────────────────────────────────────────────────────────
def fetch_ticker(symbol: str) -> pl.DataFrame:
    """Download daily OHLCV bars from yfinance, return polars DataFrame."""
    import yfinance as yf

    df = yf.download(
        symbol, start="2010-01-01", end="2024-12-31", progress=False, auto_adjust=True
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"yfinance returned empty frame for {symbol}")
    df = df.copy()
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return pl.from_pandas(df.reset_index())


def load_or_fetch(symbol: str) -> tuple[pl.DataFrame | None, str]:
    """Load from parquet cache, or download and cache."""
    cache = DATA_DIR / f"{symbol.replace('^', '').replace('.', '_')}.parquet"
    if cache.exists():
        return pl.read_parquet(cache), "cache"
    try:
        df = fetch_ticker(symbol)
        df.write_parquet(cache)
        return df, "yfinance"
    except Exception as exc:
        print(f"  {symbol} unavailable ({type(exc).__name__}: {exc})")
        return None, "failed"


def load_stock_data() -> tuple[dict[str, pl.DataFrame], str, pl.DataFrame]:
    """Load all tickers and return (stock_data, primary_symbol, primary_df)."""
    stock_data: dict[str, pl.DataFrame] = {}
    for symbol, name in TICKERS.items():
        df, source = load_or_fetch(symbol)
        if df is not None:
            stock_data[symbol] = df
            print(f"  {symbol} ({name}): {len(df)} days [{source}]")

    if "^STI" not in stock_data and "AAPL" not in stock_data:
        raise RuntimeError("Need at least ^STI or AAPL data to proceed")

    primary = "^STI" if "^STI" in stock_data else "AAPL"
    primary_df = stock_data[primary]
    print(
        f"\nPrimary: {primary} -- {len(primary_df)} days, "
        f"{primary_df['Date'].min()} -> {primary_df['Date'].max()}"
    )
    return stock_data, primary, primary_df


# ── Windowed Datasets ───────────────────────────────────────────────────
def build_dataset(
    df: pl.DataFrame,
    seq_len: int = SEQ_LEN,
    horizon: int = FORECAST_HORIZON,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    """Build (seq_len window) -> (next horizon closes) arrays with z-score normalisation.

    Returns: X, y, mean, std, n_train_windows
    """
    data = df.select(FEATURES).to_numpy().astype(np.float32)
    n = len(data)
    split_n = int(0.8 * n)
    train_data = data[:split_n]
    mean = train_data.mean(axis=0, keepdims=True)
    std = train_data.std(axis=0, keepdims=True) + 1e-8
    data_norm = (data - mean) / std

    n_windows = n - seq_len - horizon + 1
    X = np.stack([data_norm[i : i + seq_len] for i in range(n_windows)])
    y = np.stack(
        [data_norm[i + seq_len : i + seq_len + horizon, 0] for i in range(n_windows)]
    )
    split_idx = split_n - seq_len
    return X.astype(np.float32), y.astype(np.float32), mean, std, split_idx


def prepare_dataloaders(
    primary_df: pl.DataFrame,
    device: torch.device,
) -> tuple[
    DataLoader,
    DataLoader,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    np.ndarray,
    np.ndarray,
    int,
    int,
]:
    """Build train/val dataloaders and return raw tensors plus normalisation stats.

    Returns: train_loader, val_loader, X_train_t, y_train_t, X_val_t, y_val_t,
             norm_mean, norm_std, n_train_w, n_features
    """
    X_all, y_all, norm_mean, norm_std, n_train_w = build_dataset(primary_df)
    print(
        f"Built {len(X_all)} windows (seq_len={SEQ_LEN}, horizon={FORECAST_HORIZON}); "
        f"train {n_train_w}, val {len(X_all) - n_train_w}"
    )

    X_train_t = torch.from_numpy(X_all[:n_train_w]).to(device)
    y_train_t = torch.from_numpy(y_all[:n_train_w]).to(device)
    X_val_t = torch.from_numpy(X_all[n_train_w:]).to(device)
    y_val_t = torch.from_numpy(y_all[n_train_w:]).to(device)
    print(f"  X_train: {tuple(X_train_t.shape)}  y_train: {tuple(y_train_t.shape)}")
    print(f"  X_val:   {tuple(X_val_t.shape)}    y_val:   {tuple(y_val_t.shape)}")

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
    n_features = X_train_t.shape[-1]

    return (
        train_loader,
        val_loader,
        X_train_t,
        y_train_t,
        X_val_t,
        y_val_t,
        norm_mean,
        norm_std,
        n_train_w,
        n_features,
    )


# ── Experiment Tracking ─────────────────────────────────────────────────
async def _setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rnns.db"
    registry_db = "sqlite:///mlfp05_rnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    exp_name = (
        f"m5_rnns_{primary}_{experiment_suffix}"
        if experiment_suffix
        else f"m5_rnns_{primary}"
    )
    return conn, tracker, exp_name, registry, True


def setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Sync wrapper for engine setup."""
    return asyncio.run(_setup_engines(primary, experiment_suffix))


# ── Training Harness ────────────────────────────────────────────────────
def compute_gradient_norm(model: nn.Module) -> float:
    """Compute the total L2 norm of all gradients (before clipping)."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm**0.5


def _predict(model: nn.Module, x: torch.Tensor, attn: bool = False) -> torch.Tensor:
    """Forward pass, handling attention models that return a tuple."""
    out = model(x)
    return out[0] if attn else out


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Train with gradient tracking, log to ExperimentTracker."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses, gradient_norms = [], [], []
    n_params = sum(p.numel() for p in model.parameters())

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(HIDDEN_DIM),
                "seq_len": str(SEQ_LEN),
                "forecast_horizon": str(FORECAST_HORIZON),
                "epochs": str(epochs),
                "lr": str(lr),
                "clip_norm": str(clip),
                "n_params": str(n_params),
            }
        )
        print(f"  [{name}] {n_params:,} parameters")

        for epoch in range(epochs):
            model.train()
            b_losses, e_grads = [], []
            for xb, yb in train_loader:
                opt.zero_grad()
                loss = F.mse_loss(_predict(model, xb, attn), yb)
                loss.backward()
                e_grads.append(compute_gradient_norm(model))
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                opt.step()
                b_losses.append(loss.item())

            tl, gn = float(np.mean(b_losses)), float(np.mean(e_grads))
            train_losses.append(tl)
            gradient_norms.append(gn)

            model.eval()
            with torch.no_grad():
                vl = float(
                    np.mean(
                        [
                            F.mse_loss(_predict(model, xb, attn), yb).item()
                            for xb, yb in val_loader
                        ]
                    )
                )
            val_losses.append(vl)

            await run.log_metrics(
                {"train_loss": tl, "val_loss": vl, "gradient_norm": gn},
                step=epoch + 1,
            )
            print(
                f"  [{name}] epoch {epoch+1:2d}/{epochs}  "
                f"train={tl:.4f}  val={vl:.4f}  grad={gn:.4f}"
            )

        await run.log_metric("final_val_loss", val_losses[-1])

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "gradient_norms": gradient_norms,
        "final_val_loss": val_losses[-1],
    }


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Sync wrapper for training."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            device,
            epochs,
            lr,
            clip,
            attn,
        )
    )


# ── Model Registry ──────────────────────────────────────────────────────
def register_best_model(
    model: nn.Module,
    model_name: str,
    val_loss: float,
    primary: str,
    registry: ModelRegistry | None,
    has_registry: bool,
) -> None:
    """Register a model in the ModelRegistry."""
    if not has_registry or registry is None:
        print("  ModelRegistry not available, skipping registration")
        return

    model_bytes = pickle.dumps(model.state_dict())
    # Architecture/ticker are encoded in the model name; numeric facts go to MetricSpec.
    reg_result = asyncio.run(
        registry.register_model(
            name=f"m5_rnn_{model_name.lower()}_{primary.replace('^', '')}",
            artifact=model_bytes,
            metrics=[
                MetricSpec(
                    name="val_loss", value=float(val_loss), higher_is_better=False
                ),
                MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
                MetricSpec(name="seq_len", value=float(SEQ_LEN)),
                MetricSpec(name="forecast_horizon", value=float(FORECAST_HORIZON)),
                MetricSpec(name="epochs", value=float(EPOCHS)),
            ],
        )
    )
    print(f"  Registered: {reg_result.name} v{reg_result.version}")


# ── Visualisation Helpers ───────────────────────────────────────────────
def get_visualizer() -> ModelVisualizer:
    """Create a ModelVisualizer instance."""
    return ModelVisualizer()


def plot_training_curves(
    viz: ModelVisualizer,
    results: dict[str, Any],
    model_name: str,
    output_prefix: str,
) -> None:
    """Plot training/validation loss curves and gradient norms."""
    train_metrics = {
        f"{model_name} train": results["train_losses"],
        f"{model_name} val": results["val_losses"],
    }
    viz.training_history(
        metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_training_curves.html"))

    grad_metrics = {model_name: results["gradient_norms"]}
    viz.training_history(
        metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_gradient_norms.html"))


def plot_predictions(
    viz: ModelVisualizer,
    model: nn.Module,
    X_val_t: torch.Tensor,
    y_val_t: torch.Tensor,
    norm_mean: np.ndarray,
    norm_std: np.ndarray,
    output_prefix: str,
    attn: bool = False,
) -> tuple[np.ndarray, np.ndarray, torch.Tensor | None]:
    """Generate prediction-vs-actual scatter and return denormalised arrays.

    Returns: preds_denorm, actual_denorm, attn_weights (or None)
    """
    model.eval()
    with torch.no_grad():
        if attn:
            val_preds, attn_weights = model(X_val_t)
        else:
            val_preds = model(X_val_t)
            attn_weights = None

    close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
    preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
    actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean

    pred_df = pl.DataFrame(
        {
            "actual": actual_denorm[:, 0].tolist(),
            "predicted": preds_denorm[:, 0].tolist(),
        }
    )
    viz.scatter(pred_df, x="actual", y="predicted").write_html(
        str(OUTPUT_DIR / f"{output_prefix}_pred_vs_actual.html")
    )

    return preds_denorm, actual_denorm, attn_weights


def plot_time_series_overlay(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    output_prefix: str,
    title: str = "Predicted vs Actual Close Price",
    n_points: int = 200,
) -> None:
    """Plot predicted vs actual as overlaid time-series lines."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    n = min(n_points, len(preds_denorm))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        range(n), actual_denorm[:n, 0], label="Actual", color="#2196F3", linewidth=1.5
    )
    ax.plot(
        range(n),
        preds_denorm[:n, 0],
        label="Predicted",
        color="#FF5722",
        linewidth=1.5,
        linestyle="--",
        alpha=0.85,
    )
    ax.set_xlabel("Validation Window Index")
    ax.set_ylabel("Close Price")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f"{output_prefix}_time_series_overlay.png"), dpi=150)
    plt.close(fig)
    print(f"  Saved: {output_prefix}_time_series_overlay.png")


def plot_horizon_error(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    model_name: str,
) -> list[float]:
    """Print and return RMSE by forecast horizon day."""
    print(f"\n  Forecast Error by Horizon Day ({model_name}):")
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = (
            float(np.mean((preds_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        )
        rmses.append(rmse)
        print(f"    Day {day + 1}: RMSE={rmse:.2f}")
    return rmses




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3.5: Architecture Comparison — RNN vs LSTM vs GRU vs Attention
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Train all four architectures under identical conditions for fair comparison
#   - Compare accuracy, parameter efficiency, gradient health, and latency
#   - Run multi-stock generalisation tests (does the best model generalise?)
#   - Generate a comprehensive comparison dashboard with visual evidence
#   - Make architecture selection decisions for different business scenarios
#   - Register the overall best model in ModelRegistry
#
# PREREQUISITES: 01-04 (all four architecture files).
# ESTIMATED TIME: ~25-30 min
#
# DATASET: STI + APAC/global stocks via yfinance (2010-2024).
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset



## THEORY — Choosing the Right Sequence Architecture


In [ ]:
#
# By now you have built four sequence architectures. Here is when to
# use each one in production:
#
# ┌──────────────────┬─────────────────────────────────────────────────┐
# │ Architecture     │ Best For                                       │
# ├──────────────────┼─────────────────────────────────────────────────┤
# │ Vanilla RNN      │ Short sequences (<10 steps), simple patterns,  │
# │                  │ tiny models for edge/mobile deployment          │
# ├──────────────────┼─────────────────────────────────────────────────┤
# │ LSTM             │ Long sequences (20-200 steps), complex          │
# │                  │ dependencies, when you need fine-grained        │
# │                  │ memory control via forget/input/output gates    │
# ├──────────────────┼─────────────────────────────────────────────────┤
# │ GRU              │ Same tasks as LSTM but when latency or model    │
# │                  │ size matters. ~75% of LSTM parameters,          │
# │                  │ measurably faster inference.                    │
# ├──────────────────┼─────────────────────────────────────────────────┤
# │ LSTM+Attention   │ When you need explainability (which timesteps   │
# │                  │ matter?) or when different parts of the         │
# │                  │ sequence are unequally important. Healthcare,   │
# │                  │ finance, any domain with "trigger events."      │
# └──────────────────┴─────────────────────────────────────────────────┘
#
# For sequences longer than ~200 steps, Transformers (Exercise 4)
# replace all of the above with parallel self-attention.



In [ ]:
device = init_environment()



## TASK 1 — Load data and set up experiment tracking


In [ ]:
stock_data, PRIMARY, primary_df = load_stock_data()
(
    train_loader,
    val_loader,
    X_train_t,
    y_train_t,
    X_val_t,
    y_val_t,
    norm_mean,
    norm_std,
    n_train_w,
    N_FEATURES,
) = prepare_dataloaders(primary_df, device)
conn, tracker, exp_name, registry, has_registry = setup_engines(
    PRIMARY, experiment_suffix="comparison"
)



### Checkpoint 1


In [ ]:
assert X_train_t.shape[1] == SEQ_LEN
assert tracker is not None
print("--- Checkpoint 1 passed --- data and tracking ready\n")



## TASK 2 — Define all four architectures


In [ ]:
class VanillaRNN(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: Define nn.RNN layer (input_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
        # TODO: Define nn.Linear prediction head (hidden_dim, horizon)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: out, _ = self.rnn(x); return self.head(out[:, -1])
        pass
class LSTMRegressor(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: Define nn.LSTM layer (input_dim, hidden_dim, batch_first=True)
        # TODO: Define nn.Linear prediction head (hidden_dim, horizon)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: out, _ = self.lstm(x); return self.head(out[:, -1])
        pass
class GRURegressor(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: Define nn.GRU layer (input_dim, hidden_dim, batch_first=True)
        # TODO: Define nn.Linear prediction head (hidden_dim, horizon)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: out, _ = self.gru(x); return self.head(out[:, -1])
        pass
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        # TODO: Define W — nn.Linear(hidden_dim, hidden_dim)
        # TODO: Define v — nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, lstm_outputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: energy = tanh(self.W(lstm_outputs))
        # TODO: scores = self.v(energy).squeeze(-1)
        # TODO: weights = softmax(scores, dim=-1)
        # TODO: context = bmm(weights.unsqueeze(1), lstm_outputs).squeeze(1)
        # TODO: return context, weights
        pass
class LSTMWithAttention(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: Define nn.LSTM, TemporalAttention, nn.Linear head
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: lstm_out -> attention -> head
        # TODO: Return pred, attn_weights
        pass
# Instantiate all four
models = {
    "VanillaRNN": VanillaRNN(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM),
    "LSTM": LSTMRegressor(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM),
    "GRU": GRURegressor(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM),
    "LSTM+Attention": LSTMWithAttention(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM),
}
is_attn = {"VanillaRNN": False, "LSTM": False, "GRU": False, "LSTM+Attention": True}
param_counts = {}
for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    param_counts[name] = n_params
    print(f"  {name}: {n_params:,} parameters")



### Checkpoint 2


In [ ]:
assert len(models) == 4
assert param_counts["GRU"] < param_counts["LSTM"]
print("--- Checkpoint 2 passed --- all four architectures defined\n")



## TASK 3 — Train all four architectures under identical conditions


In [ ]:
print(f"\n== Training all four on {PRIMARY} (identical conditions) ==")
all_results = {}
for name, model in models.items():
    print(f"\n--- {name} ---")
    results = train_model(
        model,
        name,
        tracker,
        exp_name,
        train_loader,
        val_loader,
        device,
        attn=is_attn[name],
    )
    all_results[name] = results



### Checkpoint 3


In [ ]:
for name, res in all_results.items():
    assert len(res["train_losses"]) == EPOCHS, f"{name} should have {EPOCHS} epochs"
    assert res["final_val_loss"] < 5.0, f"{name} val loss suspiciously high"
print("\n--- Checkpoint 3 passed --- all four architectures trained\n")



## TASK 4 — Comprehensive comparison table


In [ ]:
print("=" * 80)
print(f"  {'Model':<18s} {'Params':>8s} {'Train':>8s} {'Val':>8s} {'GradNorm':>10s}")
print("-" * 80)
for name, res in all_results.items():
    print(
        f"  {name:<18s} {param_counts[name]:>8,d} "
        f"{res['train_losses'][-1]:>8.4f} {res['final_val_loss']:>8.4f} "
        f"{np.mean(res['gradient_norms']):>10.4f}"
    )
print("=" * 80)
best_name = min(all_results, key=lambda k: all_results[k]["final_val_loss"])
best_val = all_results[best_name]["final_val_loss"]
print(f"\n  Best model: {best_name} (val_loss={best_val:.4f})")



## TASK 5 — Inference latency benchmark


In [ ]:
def _predict_for_bench(model, x, attn=False):
    out = model(x)
    return out[0] if attn else out
def benchmark_inference(
    model: nn.Module, name: str, attn: bool, n_runs: int = 200
) -> float:
    model.eval()
    test_input = torch.randn(1, SEQ_LEN, N_FEATURES, device=device)
    with torch.no_grad():
        for _ in range(20):
            _predict_for_bench(model, test_input, attn)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _predict_for_bench(model, test_input, attn)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) / n_runs * 1000
    return elapsed_ms
print("\n== Inference Latency ==")
latencies = {}
for name, model in models.items():
    lat = benchmark_inference(model, name, is_attn[name])
    latencies[name] = lat
    print(f"  {name}: {lat:.3f} ms/inference")
fastest = min(latencies, key=latencies.get)
print(f"  Fastest: {fastest} ({latencies[fastest]:.3f} ms)")



## TASK 6 — Multi-stock generalisation test


In [ ]:
print(f"\n== Multi-Stock Generalisation ({best_name}) ==")
multi_stock_results: dict[str, float] = {PRIMARY: best_val}
# TODO: For each stock in stock_data (except PRIMARY):
#   - Build dataset using build_dataset()
#   - Create train/val DataLoaders
#   - Instantiate the best architecture (check best_name)
#   - Train for 8 epochs with Adam, lr=LR, clip=CLIP
#   - Evaluate val loss and store in multi_stock_results
#   - Print per-stock val loss
for symbol, sdf in stock_data.items():
    if symbol == PRIMARY or len(sdf) < SEQ_LEN + FORECAST_HORIZON + 50:
        continue
    # TODO: Build dataset, train best model, evaluate, store result
    pass
avg_cross_stock = np.mean(list(multi_stock_results.values()))
print(f"\n  Average cross-stock val loss: {avg_cross_stock:.4f}")



### Checkpoint 4


In [ ]:
assert len(multi_stock_results) >= 2, "Need multi-stock results"
print("--- Checkpoint 4 passed --- multi-stock generalisation complete\n")



## TASK 7 — Comprehensive comparison dashboard


In [ ]:
viz = get_visualizer()
# 7A: Training curves (all models overlaid)
train_metrics = {}
for label, res in all_results.items():
    train_metrics[f"{label} train"] = res["train_losses"]
    train_metrics[f"{label} val"] = res["val_losses"]
viz.training_history(
    metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
).write_html(str(OUTPUT_DIR / "05_comparison_training_curves.html"))
# 7B: Gradient norms (all models overlaid)
grad_metrics = {k: v["gradient_norms"] for k, v in all_results.items()}
viz.training_history(
    metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
).write_html(str(OUTPUT_DIR / "05_comparison_gradient_norms.html"))
# 7C: Prediction vs actual for best model
best_model = models[best_name]
best_model.eval()
with torch.no_grad():
    if is_attn[best_name]:
        val_preds, val_attn_weights = best_model(X_val_t)
    else:
        val_preds = best_model(X_val_t)
        val_attn_weights = None
close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean
pred_df = pl.DataFrame(
    {"actual": actual_denorm[:, 0].tolist(), "predicted": preds_denorm[:, 0].tolist()}
)
viz.scatter(pred_df, x="actual", y="predicted").write_html(
    str(OUTPUT_DIR / "05_comparison_pred_vs_actual.html")
)
# 7D: Comprehensive comparison figure (matplotlib)
# TODO: Create 2x3 subplot figure (20, 12) with these panels:
#   (0,0): Val loss over epochs — all 4 models overlaid with legend
#   (0,1): Gradient norms over epochs — all 4 models overlaid
#   (0,2): Parameter count vs val loss scatter (efficiency frontier)
#     - Annotate each point with model name
#   (1,0): Latency bar chart — bars coloured per model
#     - Add ms values above bars
#   (1,1): Predicted vs actual time series (best model, 150 points)
#   (1,2): Multi-stock generalisation bar chart
#     - Primary stock in model colour, others in grey
#     - Horizontal line for average cross-stock loss
#   Suptitle: f"Architecture Comparison: ... on {PRIMARY}"
#   Save to OUTPUT_DIR / "05_comparison_dashboard.png"
model_names = list(all_results.keys())
colors = {
    "VanillaRNN": "#F44336",
    "LSTM": "#2196F3",
    "GRU": "#4CAF50",
    "LSTM+Attention": "#FF9800",
}
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
# TODO: Fill in all 6 panels of the dashboard
fig.tight_layout()
fig.savefig(str(OUTPUT_DIR / "05_comparison_dashboard.png"), dpi=150)
plt.close(fig)
print("  Saved: 05_comparison_dashboard.png")
# 7E: Horizon error by day for all models
print("\n== Forecast Error by Horizon Day (all models) ==")
print(f"  {'Day':<6s}", end="")
for name in model_names:
    print(f"  {name:>16s}", end="")
print()
all_horizon_rmses = {}
for name in model_names:
    model = models[name]
    model.eval()
    with torch.no_grad():
        if is_attn[name]:
            vp, _ = model(X_val_t)
        else:
            vp = model(X_val_t)
    vp_denorm = vp.cpu().numpy() * close_std + close_mean
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = float(np.mean((vp_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        rmses.append(rmse)
    all_horizon_rmses[name] = rmses
for day in range(FORECAST_HORIZON):
    print(f"  Day {day+1:<3d}", end="")
    for name in model_names:
        print(f"  {all_horizon_rmses[name][day]:>16.2f}", end="")
    print()



### Checkpoint 5


In [ ]:
assert (OUTPUT_DIR / "05_comparison_dashboard.png").exists()
assert (OUTPUT_DIR / "05_comparison_training_curves.html").exists()
assert (OUTPUT_DIR / "05_comparison_pred_vs_actual.html").exists()
print("\n--- Checkpoint 5 passed --- comparison dashboard generated\n")



## TASK 8 — Register overall best model


In [ ]:
register_best_model(
    models[best_name],
    best_name,
    best_val,
    PRIMARY,
    registry,
    has_registry,
)



### Checkpoint 6


In [ ]:
assert best_val < 5.0, "Best model val loss should be reasonable"
print("--- Checkpoint 6 passed --- best model registered\n")



## TASK 9 — Architecture selection guide (decision framework)


Based on the experiments above, here is a decision framework:
  QUESTION 1: How long are your sequences?
    < 10 steps  -> VanillaRNN (simplest, fastest, adequate)
    10-200 steps -> LSTM or GRU (gates preserve long-range dependencies)
    > 200 steps  -> Transformer (Exercise 4 — parallel self-attention)
  QUESTION 2: Does latency matter?
    Yes (real-time, sensor, trading) -> GRU ({latencies['GRU']:.3f}ms vs LSTM {latencies['LSTM']:.3f}ms)
    No (batch, offline, training)    -> LSTM or LSTM+Attention
  QUESTION 3: Do you need explainability?
    Yes (healthcare, finance, regulated) -> LSTM+Attention (attention heatmaps)
    No (internal tool, non-regulated)    -> LSTM or GRU (simpler)
  QUESTION 4: How much data do you have?
    Small dataset (<1K sequences) -> GRU (fewer parameters, less overfitting)
    Large dataset (>10K sequences) -> LSTM+Attention (can leverage capacity)
  Results from THIS experiment on {PRIMARY}:
    Best accuracy:     {best_name} (val={best_val:.4f})
    Most efficient:    GRU ({param_counts['GRU']:,} params)
    Fastest inference: {fastest} ({latencies[fastest]:.3f}ms)
    Best gradient flow: LSTM or LSTM+Attention (gated architectures)


In [ ]:
print("=" * 80)
print("  ARCHITECTURE SELECTION GUIDE")
print("=" * 80)
print(
)



## REFLECTION


[x] Loaded data across {len(stock_data)} tickers
  [x] Built VanillaRNN, LSTM, GRU, and LSTM+Attention in torch.nn
  [x] Wrote LSTM gate equations as vectorised torch operations
  [x] Multi-step forecasting: {SEQ_LEN}-day window -> next {FORECAST_HORIZON} days
  [x] Tracked every variant with ExperimentTracker (per-epoch loss + grad norms)
  [x] Vanishing gradients: demonstrated and explained with visual evidence
  [x] Temporal attention: learnable focus over past timesteps (preview of M5.4)
  [x] Best model ({best_name}) registered in ModelRegistry
  [x] Multi-stock generalisation across {len(multi_stock_results)} tickers
  [x] Architecture selection guide for real-world decision making
  [x] Applied to Singapore business scenarios:
      - Ya Kun Kaya Toast: F&B demand forecasting (RNN)
      - SGX/DBS: Equity forecasting with prediction intervals (LSTM)
      - SMRT: Predictive maintenance for trains (GRU)
      - SGH: Clinical deterioration prediction with explainability (Attention)
  Key insight: There is no single "best" architecture. The right choice
  depends on sequence length, latency requirements, explainability needs,
  and data volume. RNNs fail on long sequences. LSTMs fix this with
  additive cell-state updates. GRUs match with fewer parameters. Attention
  lets the model choose which past steps matter. Error compounds across
  the forecast horizon.
  This exercise teaches architectures, not market timing.
  Next: Exercise 4 — Transformers replace recurrence with pure attention.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED — EXERCISE 3 COMPLETE")
print("=" * 70)
print(
)



In [ ]:
print(f"\n== Training all four on {PRIMARY} (identical conditions) ==")
all_results = {}
for name, model in models.items():
    print(f"\n--- {name} ---")
    results = train_model(
        model,
        name,
        tracker,
        exp_name,
        train_loader,
        val_loader,
        device,
        attn=is_attn[name],
    )
    all_results[name] = results
    # Per-architecture diagnostic — the comparison is the teaching
    # moment: VanillaRNN should light up CRITICAL while LSTM/GRU/
    # Attention stay HEALTHY on the same task and identical data.
    from kailash_ml import diagnose
    print(f"  ── Diagnostic Report ({name}) ──")
    report = diagnose(
        model,
        kind="dl",
        data=val_loader,
        show=False,
    )
    # ══════ EXPECTED OUTPUT (synthesized reference — side-by-side across 4 architectures) ══════
    # ┌────────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
    # │ Architecture   │ VanillaRNN   │ GRU          │ LSTM         │ Attention    │
    # ├────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤
    # │ Blood Test     │ [X] CRITICAL │ [✓] HEALTHY  │ [✓] HEALTHY  │ [✓] HEALTHY  │
    # │   min RMS      │ 8.2e-07      │ 2.7e-04      │ 2.4e-04      │ 4.1e-04      │
    # ├────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤
    # │ Saturation     │ [!] tanh 0.99│ [✓]          │ [✓]          │ [✓]          │
    # ├────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤
    # │ Stethoscope    │ [!] slow/    │ [✓] -3.1e-03 │ [✓] -2.8e-03 │ [✓] -3.6e-03 │
    # │   slope        │   oscillates │   /epoch     │   /epoch     │   /epoch     │
    # ├────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤
    # │ Final val loss │ ~3.8         │ ~1.3         │ ~1.4         │ ~0.95        │
    # │ Final val RMSE │ ~28 μg/m³    │ ~10 μg/m³    │ ~10 μg/m³    │ ~8 μg/m³     │
    # └────────────────┴──────────────┴──────────────┴──────────────┴──────────────┘
    #
    # STUDENT INTERPRETATION GUIDE — reading the comparison:
    #
    #  [BLOOD TEST — THE HISTORICAL ARC] The min RMS column is the
    #     single most important comparison in the table. 8.2e-07 →
    #     2.7e-04 is a THOUSAND-FOLD gradient preservation improvement
    #     between VanillaRNN and GRU. This IS the vanishing-gradient
    #     fix that Hochreiter posed in 1991 and that Cho (GRU) and
    #     Bengio (LSTM) solved in 2014. Slide 5T frames it: "1991-2014
    #     = 23 years between problem identification and scalable
    #     solution. The diagnostic instruments let you SEE the fix
    #     happen across architectures in a single afternoon."
    #     >> Prescription: Any time-series task with sequences >30
    #        steps MUST start with a gated architecture. VanillaRNN is
    #        a pedagogical tool, not a production option.
    #
    #  [SATURATION — TANH COLLAPSE] Only VanillaRNN shows saturation.
    #     Gated architectures use sigmoid gates to MODULATE tanh
    #     rather than to produce final output, so saturation occurs
    #     only at gate extremes (which is informative, not
    #     pathological). In contrast, VanillaRNN's output tanh
    #     saturates because there's no gating — all the hidden state
    #     flows through one tanh, which drives to ±1 on any strong
    #     signal.
    #     >> Prescription: No architectural fix — VanillaRNN
    #        fundamentally cannot be rescued on PM2.5 sequences.
    #        Choose GRU or LSTM.
    #
    #  [STETHOSCOPE — SLOPE COMPARISON] All three gated architectures
    #     converge at similar rates (~-3e-3/epoch). Attention is
    #     fastest (-3.6e-3) because the decoder gets richer per-step
    #     signal from attending to all timesteps. VanillaRNN's
    #     OSCILLATING loss (not just slow) is the pattern diagnostic:
    #     gradient explosions alternating with vanishing means
    #     gradient clipping is masking, not fixing, the underlying
    #     issue.
    #     >> Prescription: An oscillating training curve on a
    #        sequence task is never a tuning problem — it is an
    #        architecture problem.
    #
    #  [RMSE — BUSINESS IMPACT] For Singapore PM2.5 forecasting:
    #     28 μg/m³ error (VanillaRNN) means the model predicts
    #     "moderate air quality" when reality is "unhealthy" and
    #     vice versa — CLASSIFICATION FAILURES that misinform
    #     public-health messaging. 8 μg/m³ (Attention) stays WITHIN
    #     one air-quality band: predictions are actionable.
    #     >> Prescription: On public-health-adjacent tasks, accuracy
    #        is a population-health KPI. A 3x RMSE reduction (28 →
    #        8) means 3x fewer misleading alerts per year.
    #
    #  FIVE-INSTRUMENT TAKEAWAY: the comparison table IS the
    #  pedagogical payload. Four architectures, same data, same
    #  instruments — the Prescription Pad reveals WHY each wins or
    #  loses. Use this pattern throughout M5: any time you compare
    #  architectures, line them up on the five instruments, not just
    #  on final accuracy.
    # ═════════════════════════════════════════════════════════════════════



### Checkpoint 3


In [ ]:
for name, res in all_results.items():
    assert len(res["train_losses"]) == EPOCHS, f"{name} should have {EPOCHS} epochs"
    assert res["final_val_loss"] < 5.0, f"{name} val loss suspiciously high"
print("\n--- Checkpoint 3 passed --- all four architectures trained\n")



## TASK 4 — Comprehensive comparison table


In [ ]:
print("=" * 80)
print(f"  {'Model':<18s} {'Params':>8s} {'Train':>8s} {'Val':>8s} {'GradNorm':>10s}")
print("-" * 80)
for name, res in all_results.items():
    print(
        f"  {name:<18s} {param_counts[name]:>8,d} "
        f"{res['train_losses'][-1]:>8.4f} {res['final_val_loss']:>8.4f} "
        f"{np.mean(res['gradient_norms']):>10.4f}"
    )
print("=" * 80)
best_name = min(all_results, key=lambda k: all_results[k]["final_val_loss"])
best_val = all_results[best_name]["final_val_loss"]
print(f"\n  Best model: {best_name} (val_loss={best_val:.4f})")



## TASK 5 — Inference latency benchmark


In [ ]:
def _predict_for_bench(model, x, attn=False):
    out = model(x)
    return out[0] if attn else out
def benchmark_inference(
    model: nn.Module, name: str, attn: bool, n_runs: int = 200
) -> float:
    model.eval()
    test_input = torch.randn(1, SEQ_LEN, N_FEATURES, device=device)
    with torch.no_grad():
        for _ in range(20):
            _predict_for_bench(model, test_input, attn)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _predict_for_bench(model, test_input, attn)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) / n_runs * 1000
    return elapsed_ms
print("\n== Inference Latency ==")
latencies = {}
for name, model in models.items():
    lat = benchmark_inference(model, name, is_attn[name])
    latencies[name] = lat
    print(f"  {name}: {lat:.3f} ms/inference")
fastest = min(latencies, key=latencies.get)
print(f"  Fastest: {fastest} ({latencies[fastest]:.3f} ms)")



## TASK 6 — Multi-stock generalisation test


In [ ]:
print(f"\n== Multi-Stock Generalisation ({best_name}) ==")
multi_stock_results: dict[str, float] = {PRIMARY: best_val}
for symbol, sdf in stock_data.items():
    if symbol == PRIMARY or len(sdf) < SEQ_LEN + FORECAST_HORIZON + 50:
        continue
    X_s, y_s, _, _, sp = build_dataset(sdf, SEQ_LEN, FORECAST_HORIZON)
    ldr = DataLoader(
        TensorDataset(
            torch.from_numpy(X_s[:sp]).to(device),
            torch.from_numpy(y_s[:sp]).to(device),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    ldr_v = DataLoader(
        TensorDataset(
            torch.from_numpy(X_s[sp:]).to(device),
            torch.from_numpy(y_s[sp:]).to(device),
        ),
        batch_size=BATCH_SIZE,
    )
    # Train the best architecture on this stock
    if best_name == "LSTM+Attention":
        m = LSTMWithAttention(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM).to(device)
        attn_flag = True
    elif best_name == "GRU":
        m = GRURegressor(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM).to(device)
        attn_flag = False
    elif best_name == "LSTM":
        m = LSTMRegressor(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM).to(device)
        attn_flag = False
    else:
        m = VanillaRNN(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM).to(device)
        attn_flag = False
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    for _ in range(8):
        m.train()
        for xb, yb in ldr:
            opt.zero_grad()
            pred = _predict_for_bench(m, xb, attn_flag)
            F.mse_loss(pred, yb).backward()
            nn.utils.clip_grad_norm_(m.parameters(), max_norm=CLIP)
            opt.step()
    m.eval()
    with torch.no_grad():
        vl = float(
            np.mean(
                [
                    F.mse_loss(_predict_for_bench(m, xb, attn_flag), yb).item()
                    for xb, yb in ldr_v
                ]
            )
        )
    multi_stock_results[symbol] = vl
    print(f"  {symbol} ({TICKERS[symbol]}): val_loss={vl:.4f}")
avg_cross_stock = np.mean(list(multi_stock_results.values()))
print(f"\n  Average cross-stock val loss: {avg_cross_stock:.4f}")



### Checkpoint 4


In [ ]:
assert len(multi_stock_results) >= 2, "Need multi-stock results"
print("--- Checkpoint 4 passed --- multi-stock generalisation complete\n")



## TASK 7 — Comprehensive comparison dashboard


In [ ]:
viz = get_visualizer()
# 7A: Training curves (all models overlaid)
train_metrics = {}
for label, res in all_results.items():
    train_metrics[f"{label} train"] = res["train_losses"]
    train_metrics[f"{label} val"] = res["val_losses"]
viz.training_history(
    metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
).write_html(str(OUTPUT_DIR / "05_comparison_training_curves.html"))
# 7B: Gradient norms (all models overlaid)
grad_metrics = {k: v["gradient_norms"] for k, v in all_results.items()}
viz.training_history(
    metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
).write_html(str(OUTPUT_DIR / "05_comparison_gradient_norms.html"))
# 7C: Prediction vs actual for best model
best_model = models[best_name]
best_model.eval()
with torch.no_grad():
    if is_attn[best_name]:
        val_preds, val_attn_weights = best_model(X_val_t)
    else:
        val_preds = best_model(X_val_t)
        val_attn_weights = None
close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean
pred_df = pl.DataFrame(
    {"actual": actual_denorm[:, 0].tolist(), "predicted": preds_denorm[:, 0].tolist()}
)
viz.scatter(pred_df, x="actual", y="predicted").write_html(
    str(OUTPUT_DIR / "05_comparison_pred_vs_actual.html")
)
# 7D: Comprehensive comparison figure (matplotlib)
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
model_names = list(all_results.keys())
colors = {
    "VanillaRNN": "#F44336",
    "LSTM": "#2196F3",
    "GRU": "#4CAF50",
    "LSTM+Attention": "#FF9800",
}
# Panel 1: Val loss over epochs
ax = axes[0, 0]
for name in model_names:
    ax.plot(
        range(1, EPOCHS + 1),
        all_results[name]["val_losses"],
        color=colors[name],
        linewidth=2,
        label=name,
    )
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MSE Loss")
ax.set_title("Learning Curves")
ax.legend()
ax.grid(True, alpha=0.3)
# Panel 2: Gradient norms over epochs
ax = axes[0, 1]
for name in model_names:
    ax.plot(
        range(1, EPOCHS + 1),
        all_results[name]["gradient_norms"],
        color=colors[name],
        linewidth=2,
        label=name,
    )
ax.set_xlabel("Epoch")
ax.set_ylabel("Gradient L2 Norm")
ax.set_title("Gradient Health")
ax.legend()
ax.grid(True, alpha=0.3)
# Panel 3: Parameter count vs val loss (efficiency frontier)
ax = axes[0, 2]
for name in model_names:
    ax.scatter(
        param_counts[name],
        all_results[name]["final_val_loss"],
        color=colors[name],
        s=200,
        zorder=5,
        edgecolors="white",
        linewidth=2,
    )
    ax.annotate(
        name,
        (param_counts[name], all_results[name]["final_val_loss"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=9,
    )
ax.set_xlabel("Parameter Count")
ax.set_ylabel("Final Val Loss")
ax.set_title("Efficiency: Parameters vs Accuracy")
ax.grid(True, alpha=0.3)
# Panel 4: Latency comparison
ax = axes[1, 0]
bars = ax.bar(
    model_names,
    [latencies[n] for n in model_names],
    color=[colors[n] for n in model_names],
    edgecolor="white",
)
ax.set_ylabel("Inference Latency (ms)")
ax.set_title("Inference Speed")
for bar, name in zip(bars, model_names):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{latencies[name]:.3f}ms",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )
ax.grid(True, alpha=0.3, axis="y")
# Panel 5: Predicted vs actual time series (best model)
ax = axes[1, 1]
n_show = 150
ax.plot(
    range(n_show),
    actual_denorm[:n_show, 0],
    label="Actual",
    color="#2196F3",
    linewidth=1.5,
)
ax.plot(
    range(n_show),
    preds_denorm[:n_show, 0],
    label=f"{best_name} Predicted",
    color=colors[best_name],
    linewidth=1.5,
    linestyle="--",
    alpha=0.85,
)
ax.set_xlabel("Validation Window Index")
ax.set_ylabel("Close Price")
ax.set_title(f"Best Model ({best_name}): Predicted vs Actual")
ax.legend()
ax.grid(True, alpha=0.3)
# Panel 6: Multi-stock generalisation
ax = axes[1, 2]
stock_names = list(multi_stock_results.keys())
stock_losses = [multi_stock_results[s] for s in stock_names]
short_names = [
    s.replace("^", "")
    .replace(".SI", "")
    .replace(".HK", "")
    .replace(".KS", "")
    .replace(".T", "")[:6]
    for s in stock_names
]
bar_colors = [colors[best_name] if s == PRIMARY else "#78909C" for s in stock_names]
bars = ax.bar(short_names, stock_losses, color=bar_colors, edgecolor="white")
ax.axhline(
    y=avg_cross_stock,
    color="#333",
    linestyle="--",
    linewidth=1,
    label=f"Avg={avg_cross_stock:.4f}",
)
ax.set_ylabel("Validation Loss")
ax.set_title(f"Multi-Stock Generalisation ({best_name})")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
fig.suptitle(
    f"Architecture Comparison: RNN vs LSTM vs GRU vs Attention on {PRIMARY}",
    fontsize=15,
    fontweight="bold",
)
fig.tight_layout()
fig.savefig(str(OUTPUT_DIR / "05_comparison_dashboard.png"), dpi=150)
plt.close(fig)
print("  Saved: 05_comparison_dashboard.png")
# 7E: Horizon error by day for all models
print("\n== Forecast Error by Horizon Day (all models) ==")
print(f"  {'Day':<6s}", end="")
for name in model_names:
    print(f"  {name:>16s}", end="")
print()
all_horizon_rmses = {}
for name in model_names:
    model = models[name]
    model.eval()
    with torch.no_grad():
        if is_attn[name]:
            vp, _ = model(X_val_t)
        else:
            vp = model(X_val_t)
    vp_denorm = vp.cpu().numpy() * close_std + close_mean
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = float(np.mean((vp_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        rmses.append(rmse)
    all_horizon_rmses[name] = rmses
for day in range(FORECAST_HORIZON):
    print(f"  Day {day+1:<3d}", end="")
    for name in model_names:
        print(f"  {all_horizon_rmses[name][day]:>16.2f}", end="")
    print()



### Checkpoint 5


In [ ]:
assert (OUTPUT_DIR / "05_comparison_dashboard.png").exists()
assert (OUTPUT_DIR / "05_comparison_training_curves.html").exists()
assert (OUTPUT_DIR / "05_comparison_pred_vs_actual.html").exists()
print("\n--- Checkpoint 5 passed --- comparison dashboard generated\n")



## TASK 8 — Register overall best model


In [ ]:
register_best_model(
    models[best_name],
    best_name,
    best_val,
    PRIMARY,
    registry,
    has_registry,
)



### Checkpoint 6


In [ ]:
assert best_val < 5.0, "Best model val loss should be reasonable"
print("--- Checkpoint 6 passed --- best model registered\n")



## TASK 9 — Architecture selection guide (decision framework)


Based on the experiments above, here is a decision framework:
  QUESTION 1: How long are your sequences?
    < 10 steps  -> VanillaRNN (simplest, fastest, adequate)
    10-200 steps -> LSTM or GRU (gates preserve long-range dependencies)
    > 200 steps  -> Transformer (Exercise 4 — parallel self-attention)
  QUESTION 2: Does latency matter?
    Yes (real-time, sensor, trading) -> GRU ({latencies['GRU']:.3f}ms vs LSTM {latencies['LSTM']:.3f}ms)
    No (batch, offline, training)    -> LSTM or LSTM+Attention
  QUESTION 3: Do you need explainability?
    Yes (healthcare, finance, regulated) -> LSTM+Attention (attention heatmaps)
    No (internal tool, non-regulated)    -> LSTM or GRU (simpler)
  QUESTION 4: How much data do you have?
    Small dataset (<1K sequences) -> GRU (fewer parameters, less overfitting)
    Large dataset (>10K sequences) -> LSTM+Attention (can leverage capacity)
  Results from THIS experiment on {PRIMARY}:
    Best accuracy:     {best_name} (val={best_val:.4f})
    Most efficient:    GRU ({param_counts['GRU']:,} params)
    Fastest inference: {fastest} ({latencies[fastest]:.3f}ms)
    Best gradient flow: LSTM or LSTM+Attention (gated architectures)


In [ ]:
print("=" * 80)
print("  ARCHITECTURE SELECTION GUIDE")
print("=" * 80)
print(
)



## DESTINATION-FIRST CLOSE — km.diagnose


In [ ]:
# This lesson walked the journey of recurrent architectures — VanillaRNN,
# LSTM, GRU, LSTM+Attention — each with its own training loop, gradient
# norm tracking, and benchmark grid. The kailash-ml SDK ships a
# single-call diagnostic primitive that closes the production loop:
# km.diagnose inspects a trained model and emits an auto-dashboard
# (loss curves, gradient flow, dead neurons, activation stats, weight
# distributions). One cell. Every diagnostic students would otherwise
# hand-roll, ready to surface in a Plotly dashboard.
from kailash_ml import diagnose
# `kind='auto'` dispatches by model type — DLDiagnostics for torch.nn.Module.
# `data=` accepts any iterable yielding tensors; we reuse val_loader.
report = diagnose(best_model, kind="auto", data=val_loader, show=False)
report.plot_training_dashboard()
print()
print("km.diagnose: 1 line of code -> the same observability the lesson")
print("body hand-rolled in 200+ lines. This is what 'destination-first'")
print("means — when the journey is internalised, the SDK is one call.")



## REFLECTION


[x] Loaded {sum(len(df) for df in stock_data.values()):,} days across {len(stock_data)} tickers
  [x] Built VanillaRNN, LSTM, GRU, and LSTM+Attention in torch.nn
  [x] Wrote LSTM gate equations as vectorised torch operations
  [x] Multi-step forecasting: {SEQ_LEN}-day window -> next {FORECAST_HORIZON} days
  [x] Tracked every variant with ExperimentTracker (per-epoch loss + grad norms)
  [x] Vanishing gradients: demonstrated and explained with visual evidence
  [x] Temporal attention: learnable focus over past timesteps (preview of M5.4)
  [x] Best model ({best_name}) registered in ModelRegistry
  [x] Multi-stock generalisation across {len(multi_stock_results)} tickers
  [x] Architecture selection guide for real-world decision making
  [x] Applied to Singapore business scenarios:
      - Ya Kun Kaya Toast: F&B demand forecasting (RNN)
      - SGX/DBS: Equity forecasting with prediction intervals (LSTM)
      - SMRT: Predictive maintenance for trains (GRU)
      - SGH: Clinical deterioration prediction with explainability (Attention)
  Key insight: There is no single "best" architecture. The right choice
  depends on sequence length, latency requirements, explainability needs,
  and data volume. RNNs fail on long sequences. LSTMs fix this with
  additive cell-state updates. GRUs match with fewer parameters. Attention
  lets the model choose which past steps matter. Error compounds across
  the forecast horizon.
  This exercise teaches architectures, not market timing.
  Next: Exercise 4 — Transformers replace recurrence with pure attention.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED — EXERCISE 3 COMPLETE")
print("=" * 70)
print(
)

